In [1]:
import os
import glob
import re
import string

In [2]:
import pandas as pd
import emoji
import nltk
from nltk.corpus import stopwords
from spellchecker import SpellChecker

In [3]:
# Download stopwords from nltk (run once)
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nathabit/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# Define the directory containing CSV files
input_directory = "input_data"  # Change this to your actual directory

# Get all CSV file paths
csv_files = glob.glob(os.path.join(input_directory, "*.csv"))

# Read and concatenate all CSV files
df_list = [pd.read_csv(file) for file in csv_files]
final_df = pd.concat(df_list, ignore_index=True)

# Display the shape of the final DataFrame
print(f"Final DataFrame shape: {final_df.shape}")

# Optional: Save the concatenated DataFrame
final_df.to_csv("merged_data.csv", index=False)

Final DataFrame shape: (7233, 9)


/var/folders/pn/pclby9617131bypw8s71537r0000gn/T/ipykernel_57320/3982564767.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(df_list, ignore_index=True)


In [4]:
final_df['review'] = final_df['title']+" "+final_df['description']
review= final_df[['review']]
review = review.drop_duplicates()
review = review.reset_index(drop=True)
review.dropna(inplace=True)

In [5]:
review.isna().sum()

review    0
dtype: int64

In [6]:
review

,review
0,Not lasting Easily brakeble
1,Breaks easily. Not a sturdy product. Don't buy...
2,Substandard product Broke after one use. Subst...
4,Received broken 2 combs were broken. Can I get...
5,Very hard teeth Very hard teeth. It is damagin...
...,...
3330,Natural products Need some improvement as more...
3331,Heena pest Very good product
3332,Nat Habit products are just the oasis in the d...
3333,"Shiny hair Although out of 2, 1 was damaged bu..."


In [7]:
# Define the chat words dictionary
chat_words_dict = {
    "brb": "Be Right Back",
    "lol": "Laugh Out Loud",
    "rofl": "Rolling On the Floor Laughing",
    "omg": "Oh My God",
    "ttyl": "Talk To You Later",
    "idk": "I Don’t Know",
    "smh": "Shaking My Head",
    "imo": "In My Opinion",
    "imho": "In My Humble Opinion",
    "fyi": "For Your Information",
    "btw": "By The Way",
    "nvm": "Never Mind",
    "tbh": "To Be Honest",
    "gtg": "Got To Go",
    "bff": "Best Friends Forever",
    "jk": "Just Kidding",
    "ttfn": "Ta-Ta For Now",
    "yolo": "You Only Live Once",
    "afk": "Away From Keyboard",
    "irl": "In Real Life",
    "dm": "Direct Message",
    "pm": "Private Message",
    "gg": "Good Game",
    "ftw": "For The Win",
    "ppl": "People",
    "thx": "Thanks",
    "lmao": "Laughing My Ass Off",
    "icymi": "In Case You Missed It",
    "fomo": "Fear Of Missing Out",
    "omw": "On My Way",
    "idc": "I Don’t Care",
    "asap": "As Soon As Possible",
    "b/c": "Because",
    "cul8r": "See You Later",
    "gr8": "Great",
    "k": "Okay",
    "l8r": "Later",
    "np": "No Problem",
    "tmi": "Too Much Information",
    "ty": "Thank You",
    "yw": "You’re Welcome",
    "wtf": "What The Fuck",
    'u': 'you',
    'ur': 'your',
    'r': 'are',
    'b4': 'before',
    'bday': 'birthday',
    'plz': 'please',
    'cuz': 'because',
    'msg': 'message',
    'wanna': 'want to',
    'gonna': 'going to'
}

In [8]:
# Define the TextPreprocessor class
class TextPreprocessor:
    def __init__(self, chat_words_dict):
        self.chat_words_dict = chat_words_dict
        self.stop_words = set(stopwords.words('english'))
        self.spell_checker = SpellChecker()

    # Convert text to lowercase
    def to_lowercase(self, text):
        return text.lower() if isinstance(text, str) else text

    # Remove URLs
    def remove_urls(self, text):
        if isinstance(text, str):
            pattern = re.compile(r'https?://\S+|www\.\S+')
            return pattern.sub('', text)
        return text

    # Remove HTML tags
    def remove_html_tags(self, text):
        if isinstance(text, str):
            pattern = re.compile('<.*?>')
            return re.sub(pattern, '', text)
        return text

    # Replace emojis with text descriptions
    def replace_emojis(self, text):
        if isinstance(text, str):
            text_with_descriptions = emoji.demojize(text)
            # Clean up underscores and extra colons in emoji descriptions
            text_with_descriptions = text_with_descriptions.replace("_", " ").strip(":")
            return text_with_descriptions
        return text

    # Replace chat words with their full form
    def replace_chat_words(self, text):
        if isinstance(text, str):
            chat_words_re = re.compile(r'\b(' + '|'.join(self.chat_words_dict.keys()) + r')\b')
            return chat_words_re.sub(lambda x: self.chat_words_dict[x.group()], text)
        return text
    # Correct spelling using PySpellChecker
    def correct_spelling(self, text):
        if isinstance(text, str):
            words = text.split()
            corrected_words = []
            for word in words:
                if word not in self.spell_checker:
                    corrected = self.spell_checker.correction(word)
                    corrected_words.append(corrected if corrected else word)  # Ensure None is not added
                else:
                    corrected_words.append(word)
            return ' '.join(corrected_words)
        return text

    # # Correct spelling using PySpellChecker
    # def correct_spelling(self, text):
    #     if isinstance(text, str):
    #         words = text.split()
    #         corrected_words = []
    #         for word in words:
    #             # If the word is misspelled, correct it; otherwise, keep the word
    #             if word not in self.spell_checker:
    #                 corrected = self.spell_checker.correction(word)
    #                 corrected_words.append(corrected)
    #             else:
    #                 corrected_words.append(word)
    #         return ' '.join(corrected_words)
    #     return text

    # Remove stopwords
    def remove_stopwords(self, text):
        if isinstance(text, str):
            return ' '.join([word for word in text.split() if word.lower() not in self.stop_words])
        return text
    
    def remove_stopwords(self, text):
        if isinstance(text, str):
            words = text.split()
            processed_words = []
            i = 0
            while i < len(words):
                word = words[i].lower()
                if word in {"not", "no", "never"} and i + 1 < len(words):  
                    # Combine negation with the next word
                    negated_word = f"{word}_{words[i+1]}"  
                    processed_words.append(negated_word)
                    i += 2  # Skip the next word (already combined)
                elif word not in self.stop_words:
                    processed_words.append(words[i])  
                    i += 1
                else:
                    i += 1  # Skip stop word

            return ' '.join(processed_words)
    
        return text

    # Remove punctuation
    def remove_punctuation(self, text):
        if isinstance(text, str):
            return text.translate(str.maketrans('', '', string.punctuation))
        return text

    # Pipeline to apply all preprocessing steps
    def preprocess(self, text):
        text = self.to_lowercase(text)
        text = self.remove_urls(text)
        text = self.remove_html_tags(text)
        text = self.replace_emojis(text)
        text = self.replace_chat_words(text)
        text = self.correct_spelling(text)
        text = self.remove_stopwords(text)
        text = self.remove_punctuation(text)
        return text

In [9]:
# Initialize the TextPreprocessor
preprocessor = TextPreprocessor(chat_words_dict)

In [10]:
# Apply the preprocessing to the 'review_details' column
review['processed_reviews'] = review['review'].apply(preprocessor.preprocess)

In [11]:
folder_name = "artifacts"
processed_file_name = "preprocessed_data.csv"

cd = os.getcwd()

folder_path = os.path.join(cd,folder_name)

if not os.path.exists(folder_path):
    os.makedirs(folder_path)

file_path =os.path.join(folder_path,processed_file_name)
review.to_csv(file_path,index=False)

In [20]:
review.to_csv('preprocessed_data.csv',index=False)